In [1]:
import numpy as np

# Cartpole Timesteps to first solve the environment

In [2]:
def get_data_for_timesteps_table(folder_path, reward_threshold, seeds = list(range(25)), n_env=1, verbose=False):
    time_steps_first_reward_threshold = []

    for seed in seeds:
        try:
            total_rewards_seed = np.squeeze(np.load(folder_path+f"/seed_{seed}/mc_tot_returns.npy"))
            time_steps_seed = np.array([x[0] for x in np.load(folder_path+f"/seed_{seed}/time_steps_of_eval.npy")])
            tot_rewards_mean = np.array([np.mean(x) for x in total_rewards_seed])
            first_reward_threshold_index = np.where(tot_rewards_mean >= reward_threshold)[0][0]
            time_steps_first_reward_threshold.append(time_steps_seed[first_reward_threshold_index] * n_env)
        except:
            if verbose:
                print(f"Threshold did not reach in folder: {folder_path} for seed: {seed}")

    mean_timesteps = np.mean(time_steps_first_reward_threshold)
    std_timesteps = np.std(time_steps_first_reward_threshold)
    num_seeds_success = len(time_steps_first_reward_threshold)

    return mean_timesteps, std_timesteps, num_seeds_success

## Network Architecture change

In [3]:
reward_threshold = 500
num_seeds = 10
seeds = list(range(num_seeds))
algos = ["RPI", "DQN", "DoubleDQN", "PPO"]
num_neurons = [8, 16, 32, 64, 128, 256]

data = {}
for num_neuron in num_neurons:
    data[num_neuron] = {}
    for algo in algos:
        try:
            folder_path = f"results/net-arch/{algo}/width_{num_neuron}_depth_2"
            if algo == "PPO":
                n_envs = 8
            else:
                n_envs = 1
            mean_timesteps, std_timesteps, num_seeds_success = get_data_for_timesteps_table(folder_path, reward_threshold, seeds, n_envs)
            data[num_neuron][algo] = {
                "mean_timesteps": mean_timesteps,
                "std_timesteps": std_timesteps,
                "num_seeds_success": num_seeds_success
            }
        except:
            pass

print(data)

/Users/eshwar/miniforge3/envs/rpi-rl-env/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/Users/eshwar/miniforge3/envs/rpi-rl-env/lib/python3.12/site-packages/numpy/_core/_methods.py:145: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/Users/eshwar/miniforge3/envs/rpi-rl-env/lib/python3.12/site-packages/numpy/_core/_methods.py:223: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/Users/eshwar/miniforge3/envs/rpi-rl-env/lib/python3.12/site-packages/numpy/_core/_methods.py:181: RuntimeWarning: invalid value encountered in divide
  arrmean = um.true_divide(arrmean, div, out=arrmean,
/Users/eshwar/miniforge3/envs/rpi-rl-env/lib/python3.12/site-packages/numpy/_core/_methods.py:215: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


{8: {'RPI': {'mean_timesteps': np.float64(66087.5), 'std_timesteps': np.float64(24065.662129889548), 'num_seeds_success': 8}, 'DQN': {'mean_timesteps': np.float64(nan), 'std_timesteps': np.float64(nan), 'num_seeds_success': 0}, 'DoubleDQN': {'mean_timesteps': np.float64(nan), 'std_timesteps': np.float64(nan), 'num_seeds_success': 0}, 'PPO': {'mean_timesteps': np.float64(nan), 'std_timesteps': np.float64(nan), 'num_seeds_success': 0}}, 16: {'RPI': {'mean_timesteps': np.float64(35855.555555555555), 'std_timesteps': np.float64(13218.262712467265), 'num_seeds_success': 9}, 'DQN': {'mean_timesteps': np.float64(nan), 'std_timesteps': np.float64(nan), 'num_seeds_success': 0}, 'DoubleDQN': {'mean_timesteps': np.float64(nan), 'std_timesteps': np.float64(nan), 'num_seeds_success': 0}, 'PPO': {'mean_timesteps': np.float64(nan), 'std_timesteps': np.float64(nan), 'num_seeds_success': 0}}, 32: {'RPI': {'mean_timesteps': np.float64(20430.0), 'std_timesteps': np.float64(4126.027144845269), 'num_seeds_

In [4]:
print("Reward threshold: ", reward_threshold)

# 1. Define column widths
W_ALGO = 12
W_NEURON = 10
W_SEEDS = 18
W_MEAN = 18
W_STD = 16

# 2. Print the header
# < : left-align, > : right-align
header = (
    f"{'Algo':<{W_ALGO}}"
    f"{'Neurons ':>{W_NEURON}}"
    f"{f'Successful Seeds ({num_seeds})':>{W_SEEDS}}"
    f"{'Mean Timesteps':>{W_MEAN}}"
    f"{'Std Timesteps':>{W_STD}}"
)
print(header)
print("-" * len(header))

# 3. Iterate and print the data
found_data = False
for num_neuron in num_neurons:
    if num_neuron in data:
        for algo in algos:
            if algo in data[num_neuron]:
                found_data = True
                metrics = data[num_neuron][algo]
                
                # Format and print the row
                # :.2f formats floats to 2 decimal places
                print(
                    f"{algo:<{W_ALGO}}"
                    f"{num_neuron:>{W_NEURON}}"
                    f"{metrics['num_seeds_success']:>{W_SEEDS}}"
                    f"{metrics['mean_timesteps']:>{W_MEAN}.2f}"
                    f"{metrics['std_timesteps']:>{W_STD}.2f}"
                )
    print("-" * len(header))

if not found_data:
    print("No data found to display.")

Reward threshold:  500
Algo          Neurons Successful Seeds (10)    Mean Timesteps   Std Timesteps
-----------------------------------------------------------------------------
RPI                  8                 8          66087.50        24065.66
DQN                  8                 0               nan             nan
DoubleDQN            8                 0               nan             nan
PPO                  8                 0               nan             nan
-----------------------------------------------------------------------------
RPI                 16                 9          35855.56        13218.26
DQN                 16                 0               nan             nan
DoubleDQN           16                 0               nan             nan
PPO                 16                 0               nan             nan
-----------------------------------------------------------------------------
RPI                 32                10          20430.00       

# Env configs

In [5]:
reward_threshold = 500
num_seeds = 10
seeds = list(range(num_seeds))
algos = ["RPI"]

env_configs = ["g_19.6_mc_1.0_mp_0.1_l_0.5", "g_9.8_mc_0.5_mp_0.1_l_0.5", "g_9.8_mc_1.0_mp_0.1_l_0.5", "g_9.8_mc_2.0_mp_0.1_l_0.5", "g_4.9_mc_1.0_mp_0.1_l_0.5", "g_9.8_mc_1.0_mp_0.05_l_0.5", "g_9.8_mc_1.0_mp_0.2_l_0.5"]

data = {}
for env_config in env_configs:
    data[env_config] = {}
    for algo in algos:
        try:
            folder_path = f"results/env-config/{algo}/{env_config}"
            if algo == "PPO":
                n_envs = 8
            else:
                n_envs = 1
            mean_timesteps, std_timesteps, num_seeds_success = get_data_for_timesteps_table(folder_path, reward_threshold, seeds, n_envs)
            data[env_config][algo] = {
                "mean_timesteps": mean_timesteps,
                "std_timesteps": std_timesteps,
                "num_seeds_success": num_seeds_success
            }
        except:
            pass

print(data)

{'g_19.6_mc_1.0_mp_0.1_l_0.5': {'RPI': {'mean_timesteps': np.float64(nan), 'std_timesteps': np.float64(nan), 'num_seeds_success': 0}}, 'g_9.8_mc_0.5_mp_0.1_l_0.5': {'RPI': {'mean_timesteps': np.float64(nan), 'std_timesteps': np.float64(nan), 'num_seeds_success': 0}}, 'g_9.8_mc_1.0_mp_0.1_l_0.5': {'RPI': {'mean_timesteps': np.float64(nan), 'std_timesteps': np.float64(nan), 'num_seeds_success': 0}}, 'g_9.8_mc_2.0_mp_0.1_l_0.5': {'RPI': {'mean_timesteps': np.float64(nan), 'std_timesteps': np.float64(nan), 'num_seeds_success': 0}}, 'g_4.9_mc_1.0_mp_0.1_l_0.5': {'RPI': {'mean_timesteps': np.float64(nan), 'std_timesteps': np.float64(nan), 'num_seeds_success': 0}}, 'g_9.8_mc_1.0_mp_0.05_l_0.5': {'RPI': {'mean_timesteps': np.float64(nan), 'std_timesteps': np.float64(nan), 'num_seeds_success': 0}}, 'g_9.8_mc_1.0_mp_0.2_l_0.5': {'RPI': {'mean_timesteps': np.float64(nan), 'std_timesteps': np.float64(nan), 'num_seeds_success': 0}}}


In [6]:
print("Reward threshold: ", reward_threshold)

# 1. Define column widths
W_ALGO = 12
W_ENV_CONFIG = 30
W_SEEDS = 18
W_MEAN = 18
W_STD = 16

# 2. Print the header
# < : left-align, > : right-align
header = (
    f"{'Algo':<{W_ALGO}}"
    f"{'Env_Config ':>{W_ENV_CONFIG}}"
    f"{f'Successful Seeds ({num_seeds})':>{W_SEEDS}}"
    f"{'Mean Timesteps':>{W_MEAN}}"
    f"{'Std Timesteps':>{W_STD}}"
)
print(header)
print("-" * len(header))

# 3. Iterate and print the data
found_data = False
for env_config in env_configs:
    if env_config in data:
        for algo in algos:
            if algo in data[env_config]:
                found_data = True
                metrics = data[env_config][algo]
                
                # Format and print the row
                # :.2f formats floats to 2 decimal places
                print(
                    f"{algo:<{W_ALGO}}"
                    f"{env_config:>{W_ENV_CONFIG}}"
                    f"{metrics['num_seeds_success']:>{W_SEEDS}}"
                    f"{metrics['mean_timesteps']:>{W_MEAN}.2f}"
                    f"{metrics['std_timesteps']:>{W_STD}.2f}"
                )
    print("-" * len(header))

if not found_data:
    print("No data found to display.")

Reward threshold:  500
Algo                           Env_Config Successful Seeds (10)    Mean Timesteps   Std Timesteps
-------------------------------------------------------------------------------------------------
RPI             g_19.6_mc_1.0_mp_0.1_l_0.5                 0               nan             nan
-------------------------------------------------------------------------------------------------
RPI              g_9.8_mc_0.5_mp_0.1_l_0.5                 0               nan             nan
-------------------------------------------------------------------------------------------------
RPI              g_9.8_mc_1.0_mp_0.1_l_0.5                 0               nan             nan
-------------------------------------------------------------------------------------------------
RPI              g_9.8_mc_2.0_mp_0.1_l_0.5                 0               nan             nan
-------------------------------------------------------------------------------------------------
RPI      

# Penalty function

In [7]:
reward_threshold = 500
num_seeds = 10
seeds = list(range(num_seeds))
algos = ["RPI"]
penalty_functions = ["relu","cubic", "cubic_dynLambda1", "quadratic", "quadratic_dynLambda1"]

data = {}
for penalty_function in penalty_functions:
    data[penalty_function] = {}
    for algo in algos:
        try:
            folder_path = f"results/penalty/{penalty_function}"
            if algo == "PPO":
                n_envs = 8
            else:
                n_envs = 1
            mean_timesteps, std_timesteps, num_seeds_success = get_data_for_timesteps_table(folder_path, reward_threshold, seeds, n_envs)
            data[penalty_function][algo] = {
                "mean_timesteps": mean_timesteps,
                "std_timesteps": std_timesteps,
                "num_seeds_success": num_seeds_success
            }
        except:
            pass

print(data)

{'relu': {'RPI': {'mean_timesteps': np.float64(nan), 'std_timesteps': np.float64(nan), 'num_seeds_success': 0}}, 'cubic': {'RPI': {'mean_timesteps': np.float64(nan), 'std_timesteps': np.float64(nan), 'num_seeds_success': 0}}, 'cubic_dynLambda1': {'RPI': {'mean_timesteps': np.float64(nan), 'std_timesteps': np.float64(nan), 'num_seeds_success': 0}}, 'quadratic': {'RPI': {'mean_timesteps': np.float64(nan), 'std_timesteps': np.float64(nan), 'num_seeds_success': 0}}, 'quadratic_dynLambda1': {'RPI': {'mean_timesteps': np.float64(nan), 'std_timesteps': np.float64(nan), 'num_seeds_success': 0}}}


In [8]:
print("Reward threshold: ", reward_threshold)

# 1. Define column widths
W_ALGO = 12
W_PENALTY_FUNCTION = 30
W_SEEDS = 18
W_MEAN = 18
W_STD = 16

# 2. Print the header
# < : left-align, > : right-align
header = (
    f"{'Algo':<{W_ALGO}}"
    f"{'Penalty function ':>{W_PENALTY_FUNCTION}}"
    f"{f'Successful Seeds ({num_seeds})':>{W_SEEDS}}"
    f"{'Mean Timesteps':>{W_MEAN}}"
    f"{'Std Timesteps':>{W_STD}}"
)
print(header)
print("-" * len(header))

# 3. Iterate and print the data
found_data = False
for penalty_function in penalty_functions:
    if penalty_function in data:
        for algo in algos:
            if algo in data[penalty_function]:
                found_data = True
                metrics = data[penalty_function][algo]
                
                # Format and print the row
                # :.2f formats floats to 2 decimal places
                print(
                    f"{algo:<{W_ALGO}}"
                    f"{penalty_function:>{W_PENALTY_FUNCTION}}"
                    f"{metrics['num_seeds_success']:>{W_SEEDS}}"
                    f"{metrics['mean_timesteps']:>{W_MEAN}.2f}"
                    f"{metrics['std_timesteps']:>{W_STD}.2f}"
                )
    print("-" * len(header))

if not found_data:
    print("No data found to display.")

Reward threshold:  500
Algo                     Penalty function Successful Seeds (10)    Mean Timesteps   Std Timesteps
-------------------------------------------------------------------------------------------------
RPI                                   relu                 0               nan             nan
-------------------------------------------------------------------------------------------------
RPI                                  cubic                 0               nan             nan
-------------------------------------------------------------------------------------------------
RPI                       cubic_dynLambda1                 0               nan             nan
-------------------------------------------------------------------------------------------------
RPI                              quadratic                 0               nan             nan
-------------------------------------------------------------------------------------------------
RPI      